<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/10_suggest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.7.0?labpath=examples%2F10_suggest.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/10_suggest.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>

# 10 — Qubit Suggestion

`suggest_qubits` recommends a qubit count and encoding for your dataset before you build a pipeline.

- Reads feature count from the dataset
- Caps at 20 qubits by default (practical NISQ ceiling)
- Adjusts the encoding hint based on the target `task`

```bash
pip install quprep
```

In [ ]:
import numpy as np

import quprep as qd

rng = np.random.default_rng(0)
X_small = rng.uniform(0, 1, size=(100, 8))
X_wide  = rng.uniform(0, 1, size=(100, 50))

## 1. Basic suggestion

In [ ]:
s = qd.suggest_qubits(X_small)
print(s)

## 2. Task-aware encoding hints

In [ ]:
import pandas as pd

tasks = ["classification", "regression", "kernel", "qaoa", "simulation"]
rows = []
for task in tasks:
    s = qd.suggest_qubits(X_small, task=task)
    rows.append({
        "task": task,
        "n_qubits": s.n_qubits,
        "encoding_hint": s.encoding_hint,
        "nisq_safe": s.nisq_safe,
    })

pd.DataFrame(rows)

## 3. Wide dataset — NISQ ceiling kicks in

In [ ]:
s_wide = qd.suggest_qubits(X_wide)
print(f"n_features : {s_wide.n_features}")
print(f"n_qubits   : {s_wide.n_qubits}  (capped at NISQ ceiling)")
print(f"nisq_safe  : {s_wide.nisq_safe}")
print(f"\nwarning:\n  {s_wide.warning}")

## 4. Override ceiling (fault-tolerant / simulator)

In [ ]:
s_ft = qd.suggest_qubits(X_wide, max_qubits=50)
print(f"n_qubits  : {s_ft.n_qubits}")
print(f"nisq_safe : {s_ft.nisq_safe}")
print(f"warning   : {s_ft.warning}")

## 5. Apply suggestion to a pipeline

In [ ]:
s = qd.suggest_qubits(X_wide, task="classification")
print(f"Suggested n_qubits : {s.n_qubits}")
print(f"Suggested encoding : {s.encoding_hint}")

pipeline = qd.Pipeline(
    reducer=qd.PCAReducer(n_components=s.n_qubits),
    encoder=qd.AngleEncoder(),
)
result = pipeline.fit_transform(X_wide)
print(f"\nEncoded {len(result.encoded)} samples at {s.n_qubits} qubits each")

## 6. repr vs str

In [ ]:
s = qd.suggest_qubits(X_small, task="kernel")
print(f"repr : {repr(s)}")
print()
print("str:")
print(s)